# So far:

- Given p, q, and e, I can calculate n and d

- I'm happy with functionality, names, docstrings

- Speed is ???

# To Do:

- Make it actually interactive (i.e. encode a message!)

  - How to choose primes?

  - Decide good sized numbers

  - Plaintext into ASCII

     - Should I just define a class?  This seems easiest
  
  - Encrypt --> ASCII into plaintext

     - use emojis for control characters

  - Decrypt

- Decide how to distribute it

- Deal with details of distributing it

## Choosing numbers

So ASCII has 128 characters, i.e. each character is 2 hexadecimal digits i.e. 8 bits.  There are 33 control characters and 95 text characters.  I can find 33 emojis to represent the control characters, which should be okay.  (implementation will be *insane* but I'll cross that bridge when I get to it). 

So for a message that is n characters, I'll need 8n bits to encode it.  `hello, world` is 12 characters so I think 16 is a reasonable limit for message length.  If the numbers start to get too insane I'll break it up into 2 words of 8 characters.  16*8 is 128 bits (just like ascii lol) so I need to work with a modulus that is 128 bits long.

For numbers $p$ and $q$ with bitlength $j$ and $k$ respectively we expect that the bitlength of $pq$ is between $j+k-1$ and $j+k$.  This can be seen by taking $p=2^j=1000...$ and $q=2^k=1000...$.  Then the bitlength of $pq$ is clearly $j+k-1$.  

If instead $p=2^{j+1}-1=111...$ and $q=2^{k+1}-1=111...$ then the bitlength of $pq$ must be one less than the bitlength of $(p+1)(q+1)$ (except possibly in extreme cases of $p$ and $q$, which I don't care to think about) i.e. the bitlength of $pq$ is $(j+1)+(k+1)-1-1=j+k$.

Then we should choose primes that are about 65 bits long.  If we choose two 65 bit primes, the bitlength of $pq$ is guaranteed to be either 128 or 129 bits.  

So here's my main question: what is the bitlength of the traditional choice of $e$?  with $e=65537=2^{16}+1$ we must have a bit length of 17.  But that doesn't actually tell us about the size of the primes.  Well, we do know that $1<e<\lambda{pq}$ which might tell us something about the primes themselves.  But really it still only sets a lower bound (and a fuzzy one at that because $\lambda(n)$ isn't as well behaved as $\phi(n)$, from my perspective). 

Okay, instead, let's examine actual implementations of RSA.  According to wikipedia, the key length (that's the bitlength of $n$) is typically between 2048 and 4096 bits.  That means a single key can encrypt like, 256-512 ascii characters.  That's pretty good.

But at the same time, it's hard to write out a 2048 bit integer on the screen.  If $p$ is 2048 bits that implies that $\log_2(p)\approx2048$.  It follows that $$\log_{10}(p)=\frac{\log_2(p)}{\log_2(10)}\approx\frac{2048}{4}\approx512$$ so we conclude that such a prime is around 512 digits long.  

By contrast, if $p$ or $n$ is 64 bits long, its only about 16 digits long.  That's a big number but still much more managable than a 512 digit prime.  Although Victor was talking about his experience using 15 digit primes and being unable to crack them.  To be fair, I'm using the cryptosystem in the forward direction and he was trying to use it in the reverse direction, which gives me a significant advantage.  

My main concern is that for some primes in this range, we will have $\lambda(pq)<e=65537$.  I should probably check that (perhaps by seeing if there is a lower bound for $\lambda(n)$)

After some work in Google sheets, I've decided that calculating a lower bound for Carmichael's Totient is not very fun.  At least not from the perspective of finding n such that $\lambda(n)>x$ for some given x.  For example, to guarantee $\lambda(n)>36$, you have to have $n>138\,181\,680$ which is already insane.  Considering I wanted to ensure $\lambda(n)>65537$, that is infeasable.  

I couuuuuuuuuld just pick $e=17$.  Then I only need $n>65520$ which means I can use quite small primes if I want to.  17 is the last Fermat Prime before 257, which I haven't calculated the necessary n for.  

Okay so I've got a good candidate.  With $e=257$, we need to keep $n>50\,365\,204\,798\,708\,800$.  That's not horrible.  That ensures that $\lambda(n)>e$.  If you're curious, $\lambda(50\,365\,204\,798\,708\,800)=240$.  

So then what must our key length be? how many bits does it take to represent 50 quadrillion? we can find that by rounding up to the next power of 2.  

In [8]:
print(2**56)

72057594037927936


In [9]:
print(2**56 // 1_000_000_000_000_000)

72


So our key length would be 56 bits at a minimum.  That means (since we were aiming for a key length of ~128 bits anyway) we can choose $e=257$ and not worry about $\lambda(n)<e$

I'm not 100% sure, but I'm guessing that our primes probably shouldnt be anywhere less than 56 bits either.  So maybe we'll select two primes between 2^63 and 2^65.  